In [18]:
import os
import numpy as np
import pandas as pd
import scipy
import statsmodels
import sklearn

In [25]:
arquivo_entrada = r"C:\_Master\programacao\Guilherme-Betta\IC2\data\out\knn.xlsx"

dados = pd.read_excel(arquivo_entrada)

metadados = ['Data', 'Estatistica', 'data_normalizada']

variaveis = [coluna for coluna in dados.select_dtypes(include=float)]

# dados[variaveis].head()
dados.select_dtypes(exclude=float)

,Unnamed: 0.1,Unnamed: 0,Data,data_normalizada
0,0,0,2009-01-01,2009-01-01
1,1,1,2009-02-01,2009-02-01
2,2,2,2009-03-01,2009-03-01
3,3,3,2009-04-01,2009-04-01
4,4,4,2009-05-01,2009-05-01
...,...,...,...,...
187,187,187,2024-08-01,2024-08-01
188,188,188,2024-09-01,2024-09-01
189,189,189,2024-10-01,2024-10-01
190,190,190,2024-11-01,2024-11-01


In [26]:
dados[variaveis].isna().sum()

COR_Max.              0
COR_Med.              0
COR_Min.              0
TURB._Max.            0
TURB._Med.            0
TURB._Min.            0
pH_Max.               0
pH_Med.               0
pH_Min.               0
ALC._Max.             0
ALC._Med.             0
ALC._Min.             0
AC._Max.              0
AC._Med.              0
AC._Min.              0
O.C._Max.             0
O.C._Med.             0
O.C._Min.             0
O.D._Max.             0
O.D._Med.             0
O.D._Min.             0
Cl_Max.               0
Cl_Med.               0
Cl_Min.               0
DUR._Max.             0
DUR._Med.             0
DUR._Min.             0
Fe_Max.               0
Fe_Med.               0
Fe_Min.               0
Mn_Max.               0
Mn_Med.               0
Mn_Min.               0
Cond._Max.            0
Cond._Med.            0
Cond._Min.            0
Cianobacteria_Max.    0
Cianobacteria_Med.    0
Cianobacteria_Min.    0
C.F._Max.             0
C.F._Med.             0
C.F._Min.       

# Detecção de outliers

## Métodos univariados

### Métodos estatísticos

Z-Score, IQR, Z-Test

#### Z-Score

In [27]:
from scipy.stats import zscore

for coluna in dados[variaveis]:
    dados[variaveis].apply(zscore)

In [35]:
# Calcular Z-Scores para todas as colunas numéricas
z_scores = zscore(dados[variaveis])

# Identificar outliers: |z| > 3 (threshold comum)
threshold = 3
outliers_zscore = (np.abs(z_scores) > threshold)

# Contar outliers por coluna
outliers_por_coluna_z = outliers_zscore.sum(axis=0)
print("Outliers detectados por Z-Score (|z| > 3):")
for col, count in zip(variaveis, outliers_por_coluna_z):
    print(f"{col}: {count}")

total_outliers_z = outliers_zscore.any(axis=1).sum()
print(f"\nTotal de linhas com pelo menos um outlier (Z-Score): {total_outliers_z}")

Outliers detectados por Z-Score (|z| > 3):
COR_Max.: 4
COR_Med.: 3
COR_Min.: 4
TURB._Max.: 4
TURB._Med.: 3
TURB._Min.: 4
pH_Max.: 0
pH_Med.: 3
pH_Min.: 1
ALC._Max.: 5
ALC._Med.: 4
ALC._Min.: 3
AC._Max.: 5
AC._Med.: 4
AC._Min.: 5
O.C._Max.: 4
O.C._Med.: 3
O.C._Min.: 2
O.D._Max.: 1
O.D._Med.: 0
O.D._Min.: 3
Cl_Max.: 4
Cl_Med.: 3
Cl_Min.: 4
DUR._Max.: 3
DUR._Med.: 2
DUR._Min.: 1
Fe_Max.: 2
Fe_Med.: 4
Fe_Min.: 7
Mn_Max.: 4
Mn_Med.: 7
Mn_Min.: 4
Cond._Max.: 3
Cond._Med.: 3
Cond._Min.: 3
Cianobacteria_Max.: 3
Cianobacteria_Med.: 3
Cianobacteria_Min.: 4
C.F._Max.: 5
C.F._Med.: 6
C.F._Min.: 4
Clorofila_Max.: 2
Clorofila_Med.: 2
Clorofila_Min.: 2
F_Max.: 3
F_Med.: 0
F_Min.: 3

Total de linhas com pelo menos um outlier (Z-Score): 52


#### IQR

In [36]:
# Calcular IQR e detectar outliers usando método de Tukey
def detectar_outliers_iqr(df, coluna):
    Q1 = df[coluna].quantile(0.25)
    Q3 = df[coluna].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    outliers = (df[coluna] < limite_inferior) | (df[coluna] > limite_superior)
    return outliers.sum(), limite_inferior, limite_superior

print("Outliers detectados por IQR (Tukey):")
total_outliers_iqr = 0
for coluna in variaveis:
    count, lim_inf, lim_sup = detectar_outliers_iqr(dados, coluna)
    print(f"{coluna}: {count} outliers (limites: {lim_inf:.3f} a {lim_sup:.3f})")
    total_outliers_iqr += count

print(f"\nTotal de outliers (IQR, somando todas as colunas): {total_outliers_iqr}")

# Também podemos contar linhas com pelo menos um outlier
outliers_por_linha_iqr = np.zeros(len(dados), dtype=bool)
for coluna in variaveis:
    # Melhor calcular por coluna e combinar
    Q1 = dados[coluna].quantile(0.25)
    Q3 = dados[coluna].quantile(0.75)
    IQR = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    outliers_por_linha_iqr |= (dados[coluna] < lim_inf) | (dados[coluna] > lim_sup)

total_linhas_outliers_iqr = outliers_por_linha_iqr.sum()
print(f"Total de linhas com pelo menos um outlier (IQR): {total_linhas_outliers_iqr}")

Outliers detectados por IQR (Tukey):
COR_Max.: 12 outliers (limites: -345.250 a 796.750)
COR_Med.: 7 outliers (limites: -91.125 a 297.875)
COR_Min.: 15 outliers (limites: -15.250 a 110.750)
TURB._Max.: 16 outliers (limites: -450.000 a 808.000)
TURB._Med.: 14 outliers (limites: -105.500 a 202.500)
TURB._Min.: 20 outliers (limites: -11.500 a 32.500)
pH_Max.: 16 outliers (limites: 7.100 a 7.900)
pH_Med.: 4 outliers (limites: 6.900 a 7.700)
pH_Min.: 4 outliers (limites: 6.550 a 7.750)
ALC._Max.: 7 outliers (limites: -0.500 a 147.500)
ALC._Med.: 7 outliers (limites: 4.375 a 117.375)
ALC._Min.: 7 outliers (limites: 4.000 a 92.000)
AC._Max.: 25 outliers (limites: -0.500 a 27.500)
AC._Med.: 22 outliers (limites: 1.359 a 19.068)
AC._Min.: 17 outliers (limites: 0.875 a 13.875)
O.C._Max.: 12 outliers (limites: 0.194 a 22.044)
O.C._Med.: 5 outliers (limites: 2.750 a 13.550)
O.C._Min.: 4 outliers (limites: 1.700 a 10.500)
O.D._Max.: 5 outliers (limites: 1.712 a 7.813)
O.D._Med.: 4 outliers (limites

#### Z-Test

Para ser útil, precisa-se de valores médios para cada variável para colocar no argumento "value=" do método utilizado

In [37]:
from statsmodels.stats.weightstats import ztest
z_test = ('test_statistic', 'p_value')
for coluna in variaveis:
    z_test = ztest(dados[coluna], value=np.mean(dados[coluna]))
    #if :    # Implementar alguma lógica que busque comparar o "p_value" com resultado
            #, de forma que, se o p_value < 0,05, haja um print relatando que os resultados
            # da variável estão diferentes do habitual
    print(z_test)
    

(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np.float64(1.0))
(np.float64(0.0), np

### Métodos de Regressão
NAIVE,
ARIMA

## Métodos Multivariados Não Supervisionados
LOF, STRAY, Distance From the Mean

### LOF

In [34]:
from sklearn.neighbors import LocalOutlierFactor

# Aplicar LOF nas colunas numéricas
lof = LocalOutlierFactor(n_neighbors=20, contamination='auto')

outlier_labels = lof.fit_predict(dados[variaveis])

outlier_scores = lof.negative_outlier_factor_

# Adicionar coluna com rótulos de outliers ao DataFrame
dados['outlier_lof'] = outlier_labels

# Número de outliers detectados
num_outliers = (outlier_labels == -1).sum()
print(f"Número de outliers detectados pelo LOF: {num_outliers}")

# Exibir algumas estatísticas dos scores
print(f"Score LOF mínimo: {outlier_scores.min():.3f}")
print(f"Score LOF máximo: {outlier_scores.max():.3f}")
print(f"Média dos scores LOF: {outlier_scores.mean():.3f}")

Número de outliers detectados pelo LOF: 42
Score LOF mínimo: -5.101
Score LOF máximo: -0.932
Média dos scores LOF: -1.333


# Transformação